In [ ]:
import ast
import os
import ipynbname
import pandas as pd
import numpy as np
from Functions.AutoCloud_V4_5 import *   # V4_5 contém o V4_4 (rls_mode='log' é o padrão)
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.TedaOptimize_V4_5 import Optimize, OptimizeRho, RunTag
import copy

FileName = ipynbname.name()

# Configuração do experimento
FEEDBACK = 'symmetric'      # 'symmetric' (gradiente no passo atual) ou 'random' (ablação: RFLO + DFA)
WTAG, WTAP = False, False   # configuração principal: soma ponderada dos grânulos (eq. 36)
EOL = 0.3                   # limiar de fim de vida (EoL) do HI
W = 1                       # janela do DSI em cada X_k (nº de amostras do RS; PrepareData/WindowDSI)

LAM = 1.0                   # fator de esquecimento (1.0 = todos os passos com o mesmo peso)

out_path = os.path.join('Optimization', RunTag(FileName, FEEDBACK, WTAG, WTAP), 'multi', 'Optimization.csv')
dfs = [pd.read_csv(out_path)] if os.path.exists(out_path) else []
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_excel(r'Dataset\HI.xlsx')
sig = HI['PC1'].values

In [ ]:
params = [3.5,28,[70],8,0,1,2,5,19,0]        # [m, nI, nR, nO (futuros de Z_k), mO (passados de Z_k, incl. h_k), N1, N2, N3, τ, act]
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX)   # avisa se o fluxo não alcança o EoL
if isinstance(nR, str): nR = ast.literal_eval(nR)


# RUL intervalar (RULInterv2): protocolo 3 na janela de entrada + análise intervalar da rede (pesos numéricos)
RHO = 0.76                  # rho inicial: múltiplo do erro estimado do HI (eP) somado/subtraído aos valores previstos
# rho dinâmico (sugestão 3): subida de gradiente online em V = média(cov*sp) do HI
RHO_DYN = True              # False: rho estático (= RHO)
RHO_ETA, RHO_T, RHO_BETA = 0.51, 0.51, 0.59   # passo, temperatura da cobertura suavizada (escore |e|/e_est), momento
RHO_MIN, RHO_MAX = 1e-2, 10
INTR_STATE = 'reset'        # estado recorrente intervalar: 'reset' (só a incerteza da janela do passo) ou 'carry' (propagado)

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               ff_rls=0.91,dt_rls=27
               )

for j,_ in enumerate(X[:J]):

    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=L,show=False)
    teda.WAPE_HI2(Y[j],Z[j])
    teda.Adapt(Y[j],Z[j])
    teda.UpdateRLS(mode='HI')

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f}")
PlotDSI_3D_PLY(teda,xi=10, yf=100)


In [ ]:
print(df_rho)

In [7]:
params = [3.5,28,[70],8,0,1,2,5,19,0]        # [m, nI, nR, nO (futuros de Z_k), mO (passados de Z_k, incl. h_k), N1, N2, N3, τ, act]
df_rho = OptimizeRho(
        RS=RS,HI=HI,eol=EOL,w=W,lam=LAM,
        FileName=None,params=params,OptSampler='random',OptSeed=None,
        n_study=2,timeout=None,n_trials=1000,patience=350,
        rhoS=None,etaS=None,TS=None,betaS=None,ffS=None,dtS=None,
        rho_min=1e-2,rho_max=10,intr_state='reset',Vtarget='HI',rls_mode='log',
        feedback=FEEDBACK,wtaG=WTAG,wtaP=WTAP,
        OptReset=False,OptWarm=False,n_best=5)

OPTIMIZATION PROGRESS
Layers (nL): 1  |  Session: 2/2 (rho_HI_92f84112_s2)  |  Sampler: Random  |  Config: None_symmetric_G0P0
Trials in this study: 197 stored before this run  |  This run: 350/1000  |  Elapsed: 00:05:02 / 00:10:00

BEST TRIAL
Trial #096 (sessões anteriores) | 1-V: 0.0139 | Params: {'rho': 0.7657731591564376, 'eta': 0.0029618425037475477, 'T': 0.17581405713009118, 'beta': 0.9236462671047664, 'ff_rls': 0.9498340653815807, 'dt_rls': 31647.129966094315}

LAST 3 TRIALS
Trial #544 [COMPLETE] | Score: 0.0356 | Params: {'rho': 1.4725033620670818, 'eta': 0.4098897566326235, 'T': 0.10862914666729345, 'beta': 0.98277572866924, 'ff_rls': 0.8535428755600691, 'dt_rls': 9272.275721349693}
Trial #545 [COMPLETE] | Score: 0.0426 | Params: {'rho': 0.08219617643939227, 'eta': 0.0585580839886724, 'T': 0.06524175074695926, 'beta': 0.5376152606134836, 'ff_rls': 0.780297667046683, 'dt_rls': 190.46566935777537}
Trial #546 [COMPLETE] | Score: 0.0264 | Params: {'rho': 0.2969825171817985, 'eta':

In [8]:
print(df_rho)

          V       rho       eta         T      beta    ff_rls         dt_rls
0  0.986200  1.447561  0.163343  0.113474  0.439462  0.934997   54523.701244
1  0.986173  2.861021  0.255613  1.629019  0.224415  0.891275   77652.688251
2  0.986113  0.765773  0.002962  0.175814  0.923646  0.949834   31647.129966
3  0.986089  0.949481  0.011348  0.083714  0.069306  0.944770  637079.801472
4  0.986087  0.948136  0.003206  0.126620  0.789815  0.933972     664.928535


In [10]:
rho_params = df_rho.iloc[0,1:].values
RHO, RHO_ETA, RHO_T, RHO_BETA, ff_rls, dt_rls = rho_params

params = [3.5,28,[70],8,0,1,2,5,19,0]        # [m, nI, nR, nO (futuros de Z_k), mO (passados de Z_k, incl. h_k), N1, N2, N3, τ, act]
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX)   # avisa se o fluxo não alcança o EoL
if isinstance(nR, str): nR = ast.literal_eval(nR)


RHO_DYN = True              # False: rho estático (= RHO)
RHO_MIN, RHO_MAX = 1e-2, 10
INTR_STATE = 'reset'        # estado recorrente intervalar: 'reset' (só a incerteza da janela do passo) ou 'carry' (propagado)

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               ff_rls=ff_rls,dt_rls=dt_rls
               )

for j,_ in enumerate(X[:J]):

    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=L,show=False)
    teda.WAPE_HI2(Y[j],Z[j])
    teda.Adapt(Y[j],Z[j])
    teda.UpdateRLS(mode='HI')

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f}")
PlotDSI_3D_PLY(teda,xi=10, yf=100, HideCol=[1,2,4])


HI (HI_Granularity): cov=0.942 | sp=0.872 | V=média(cov*sp)=0.8426


In [ ]:
teda.V

In [ ]:
# Etapa 1: modelo (predições pontuais de HI e RUL, rho fixo)
# df_opt = Optimize(RS=RS, HI=HI, eol=EOL, w=W, lam=LAM, FileName=FileName, OptDim=1, OptSampler='TPE', OptSeed=0, n_study=1, n_trials=500, timeout=3600, nLS=[1,1])
# Etapa 2: rho e parâmetros do rho dinâmico (maximiza V) no melhor modelo da etapa 1 (ou params=[...])
# df_rho = OptimizeRho(RS=RS, HI=HI, eol=EOL, w=W, lam=LAM, FileName=FileName, params=None, OptSampler='TPE', OptSeed=0, n_trials=100, timeout=3600, Vtarget='HI')